# NumPy — Ejercicios Avanzados (Sesión Tutorial)
**Carrera:** Ciencia de Datos e Inteligencia Artificial  
**Asignatura:** Programación II  


> Objetivo: practicar *broadcasting* avanzado, álgebra lineal, `einsum/tensordot`, indexación booleana, simulación y rendimiento, con foco en soluciones vectorizadas (sin bucles explícitos).


In [1]:
# Configuración
import numpy as np
import math, time

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(2025)
print("NumPy:", np.__version__)

NumPy: 2.3.5


## 1) Broadcasting avanzado
Trabaja únicamente con operaciones vectorizadas y `keepdims=True` donde corresponda.


In [2]:
# 1.1 Estandarización por columnas (z-score) con broadcasting
X = rng.normal(loc=10, scale=3, size=(200, 5))

# TODO: calcula Z = (X - media_col) / std_col a lo largo del eje 0
media_col = X.mean(axis = 0, keepdims = True) # shape (1, 5)
std_col   = X.std(axis = 0, ddof = 0, keepdims = True) # shape (1, 5)
Z = (X - media_col) / std_col
# Verifica media≈0 y std≈1 por columna
assert np.allclose(Z.mean(axis=0), 0, atol=1e-12)
assert np.allclose(Z.std(axis=0, ddof=0), 1, atol=1e-12)
Z.shape

(200, 5)

In [3]:
# 1.2 Distancias euclidianas por pares (matriz NxN) sin bucles
Y = rng.normal(size=(150, 3))  # 150 puntos en R^3
# TODO: matriz D donde D[i,j] = ||Y[i]-Y[j]||_2
# Pista: usa ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b y luego sqrt
# ||Y[i]||^2 para cada punto i
sq_norm = np.sum(Y**2, axis = 1, keepdims = True) # shape (150, 1)

# Matriz Gram: G[i,j] = Y[i]·Y[j].
G = Y @ Y.T # gram matrix (Y @ Y.T)   shape (150,150)

# ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b
# Broadcasting: sq_norm (150,1) + sq_norm.T (1,150) - 2*G (150,150)
D2 = sq_norm + sq_norm.T - 2 * G # distancias cuadradas

# Asegurar que valores numéricamente negativos sean cero.
D2[D2 < 0] = 0

# Raíz cuadrada para obtener distancias
D = np.sqrt(D2) # distancias

# checks
assert D.shape == (150,150)
assert np.all(D >= -1e-12)  # tolerancia numérica
D[:3, :3]

array([[0.    , 1.5263, 1.6535],
       [1.5263, 0.    , 0.6364],
       [1.6535, 0.6364, 0.    ]])

In [4]:
# 1.3 Softmax estable a lo largo del último eje en un tensor 3D
T = rng.normal(size=(4, 5, 6))
# TODO: aplica softmax estable por eje -1 (colapsa la última dimensión)
# pista: restar el máximo por fila antes de exponentes
# Encuentra el maximo a lo largo del ultimo eje, manteniendo las dimensiones.
shift = T.max(axis = -1, keepdims = True) # shape (4,5,1)

# Resta el maximo para la estabilidad numerica y aplica el exponente.
exps = np.exp(T - shift) # shape (4,5,6).

# Divide por la suma a lo largo del ultimo eje.
soft = exps / exps.sum(axis = -1, keepdims = True) # shape (4,5,6).

# Verifica que cada vec. a lo largo del último eje suma 1
assert np.allclose(soft.sum(axis=-1), 1, atol=1e-9)
soft.shape

(4, 5, 6)

## 2) Álgebra lineal
Usa `np.linalg` de forma robusta y verifica con residuos/igualdades.


In [ ]:
# 2.1 Resolver sistema Ax=b y verificar residuo
A = rng.normal(size=(6,6))
b = rng.normal(size=(6,))

# x = solución de Ax=b (usa solve)
x = np.linalg.solve(A, b) # Resuelve el sistema lineal Ax = b.

residuo = np.linalg.norm(A@x - b)
assert residuo < 1e-10
print(f"Residuo: {residuo}")

Residuo: 2.7755575615628914e-16


In [6]:
# 2.2 Autovalores/vectores en matriz simétrica
S = rng.normal(size=(5,5)); S = (S + S.T)/2  # simetriza

# usa eigh (no eig) y verifica S*v ≈ λ*v
vals, vecs = np.linalg.eigh(S) # Para matrices simetricas.
check = np.linalg.norm(S @ vecs - vecs * vals)
assert check < 1e-10
vals[:3]

array([-1.9077, -1.4409, -0.4647])

In [25]:
# 2.3 Aproximación de rango bajo con SVD (k=2)
M = rng.normal(size=(30, 10))
# SVD y aproximación M_k con k=2
U, s, Vt = np.linalg.svd(M, full_matrices = False) # Svd compacta sin 0.
k = 2
Mk = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :] # U[:,:k] @ diag(s[:k]) @ Vt[:k]
err = np.linalg.norm(M - Mk) / np.linalg.norm(M)
err.item()

0.7932853272951167

## 3) `einsum` y `tensordot`
Formula contracciones tensoriales de forma explícita.


In [8]:
# 3.1 Producto batched de matrices con einsum
# batch B=8, matrices 4x5 y 5x3 -> resultado 4x3 por batch
A = rng.normal(size=(8,4,5))
B = rng.normal(size=(8,5,3))
# C[b] = A[b] @ B[b] con einsum
C = np.einsum('b i j, b j k -> b i k', A, B) # einsum string
assert C.shape == (8,4,3)
C.shape

(8, 4, 3)

In [9]:
# 3.2 Tensordot equivalente y verificación
C_td = np.tensordot(A, B, axes = ([2], [1])) # tensordot con axes apropiados
# Ahora C_td tiene una forma (8,4,8,3) ya que tensordot no fusiona los ejes batch.
# Se necesita extraer la diagonal a lo largo de los ejes batch.
C_td = np.einsum('b i b k -> b i k', C_td) 

assert np.allclose(C, C_td)
C_td.shape

(8, 4, 3)

In [10]:
# 3.3 Outer product + reducción: suma_i x_i * y_i * z (broadcasting)
x = rng.normal(size=(50,))
y = rng.normal(size=(50,))
z = rng.normal(size=(10,))         # vector independiente
# Objetivo: un tensor T[j] = sum_i x_i * y_i * z_j  (resultado de shape (10,))
# usa einsum o broadcasting eficiente (sin bucles)
T = np.einsum('i, i, j -> j', x, y, z)
print(T.shape)
print(T[:5]) # Primeros 5 valores

(10,)
[ 1.633  -1.0961  1.9034 -1.3206 -0.4676]


## 4) Indexación booleana y agregaciones tipo *groupby*
Evita bucles; usa `np.where`, `np.add.at`, `np.bincount` o `take_along_axis`.


In [26]:
# 4.1 Reemplazo de outliers por mediana (z-score)
X = rng.normal(size=1000)
X[::50] = 10  # inyecta outliers

# calcula z = (X - mean)/std y reemplaza |z|>3 por la mediana de X sin outliers
mu = X.mean() # media de todo X
sd = X.std() # desviacion estandar de todo X
z = (X - mu) / sd # z - scores
mask = np.abs(z) > 3 # booleano de outliers
med = np.median(X[~mask]) # mediana de X[~mask]

X_clean = np.where(mask,med, X) # reemplaza outliers por mediana.
float(np.mean(np.abs((X_clean - X_clean.mean())/X_clean.std()) > 3))  # proporción outliers post-limpieza

0.002

In [12]:
# 4.2 Agregación por claves enteras (tipo groupby) con np.add.at
keys = rng.integers(0, 5, size=200)   # grupos 0..4
vals = rng.normal(size=200)
# suma por grupo (vectorizado)
sums = np.zeros(5)
# pista: np.add.at(sums, keys, vals)
np.add.at(sums, keys, vals)

# cuenta por grupo y promedio por grupo
counts = np.bincount(keys, minlength=5)
means = sums/counts # Promedio por grupo.

sums, counts, means

(array([  9.3856, -20.185 ,  -0.8559,   3.7128,   4.5406]),
 array([40, 41, 37, 44, 38]),
 array([ 0.2346, -0.4923, -0.0231,  0.0844,  0.1195]))

## 5) Convolución 1D y correlación cruzada
Usa `np.convolve` y compara con implementación vía `einsum` para ventanas.


In [13]:
# 5.1 Promedio móvil y comparación con vectorización
s = rng.normal(loc=0, scale=1, size=100)
w = np.ones(7)/7  # ventana de 7

# promedio móvil con np.convolve (modo 'valid')
mov = np.convolve(s, w, mode = 'valid') # Forma (100-7+1) = 94.

# implementación alternativa con strides/einsum (solo fórmula; puedes usar einsum)
# Sugerencia de shape final: 94 (100-7+1)
# Hints: crea una matriz de ventanas con broadcasting y aplica einsum 'ij, j -> i'
# (opcional; si no lo haces, mantén sólo mov)
from numpy.lib.stride_tricks import sliding_window_view
windows = sliding_window_view(s, 7)
mov_alt = np.einsum('ij, j -> i', windows, w) # Promedio ponderado.

mov[:5]

array([-0.1908,  0.1941, -0.0758, -0.2756, -0.3828])

## 6) Rendimiento: `np.where` vs `np.vectorize`
Evita `np.vectorize` cuando sea posible. Compara tiempos (aproximados).


In [14]:
# 6.1 Comparación simple de tiempos
arr = rng.normal(size=5_000_000)

def f_scalar(x):
    # función pieza a trozos: |x| si |x|<1; x^2 en caso contrario
    return abs(x) if abs(x) < 1 else x*x

t0 = time.time()
out_where = np.where(np.abs(arr) < 1, np.abs(arr), arr*arr)
t_where = time.time() - t0

vec_f = np.vectorize(f_scalar)
t0 = time.time()
out_vec = vec_f(arr)
t_vec = time.time() - t0

print(f"np.where: {t_where:.4f}s, np.vectorize: {t_vec:.4f}s")
assert np.allclose(out_where, out_vec)

np.where: 0.1148s, np.vectorize: 0.9149s


## 7) Simulación Monte Carlo: estimar π
Genera N pares (x,y) ~ U[0,1] y cuenta cuántos caen en el cuarto de círculo.


In [27]:
# 7.1 Estimador de π (vectorizado)
N = 2_000_000
xy = rng.random((N,2))
inside = (xy[:, 0]**2 + xy[:, 1]**2) <= 1 # condición x^2 + y^2 <= 1

# Estimacion de pi.
pi_hat = 4 * inside.sum()/N
pi_hat.item()

3.141496

## 8) Bonus: una iteración de K-means (vectorizada)
Dados datos `X` (N×d) y centros `C` (k×d), asigna cada punto a su centro más cercano y recalcula centros.


In [16]:
# 8.1 Una iteración de K-means
X = rng.normal(size=(500, 2))
C = rng.normal(size=(4, 2))  # k=4

# asigna etiquetas usando distancias cuadradas con broadcasting
# dist2[n,k] = ||X[n]-C[k]||^2
diff = X[:, np.newaxis, :] - C[np.newaxis, :, :]
dist2 = np.sum(diff**2, axis = 2) # shape (500,4)
labels = np.argmin(dist2, axis = 1) # argmin por fila -> shape (500,)

# recalcula nuevos centros como promedio por cluster (usa bincount/add.at)
C_new = np.zeros_like(C)
counts = np.bincount(labels, minlength=C.shape[0])  # (4,)

# Suma por cluster
for d in range(X.shape[1]):
    np.add.at(C_new[:,d], labels, X[:,d])
    
C_new = C_new / counts[:, None]
C_new, counts

(array([[ 0.1586,  0.4149],
        [ 0.3756,  1.4796],
        [ 0.6569, -0.9012],
        [-1.0846, -0.1407]]),
 array([111,  67, 190, 132]))

<details>
<summary><strong>Pistas rápidas</strong></summary>

- **1.1** Usa `mean(axis=0, keepdims=True)` y `std(axis=0, keepdims=True)` (evita `ddof=1`).
- **1.2** `sq_norm = (Y**2).sum(axis=1, keepdims=True)` y `G = Y @ Y.T`.
- **1.3** `shift = T.max(axis=-1, keepdims=True)`; `soft = exps / exps.sum(axis=-1, keepdims=True)`.
- **2.1** `np.linalg.solve(A,b)`.
- **2.2** `np.linalg.eigh(S)`.
- **2.3** `U,s,Vt = np.linalg.svd(M, full_matrices=False)` y luego `U[:,:k] @ (s[:k,None]*Vt[:k])`.
- **3.1** `einsum('bij,bjk->bik', A, B)`.
- **3.2** `np.tensordot(A, B, axes=([2],[1]))`.
- **3.3** `T = z * (x @ y)` o `einsum('i,i,j->j', x, y, z)`.
- **4.1** `np.where(mask, med, X)` tras calcular `mask = np.abs(z)>3`.
- **4.2** `np.add.at(sums, keys, vals)`; `means = sums / counts` con cuidado de ceros.
- **5.1** `np.convolve(s, w, mode='valid')`.
- **6.1** Evita medir con `%timeit` aquí; usa `time.time()` como en el ejemplo.
- **7.1** `inside = (xy**2).sum(axis=1) <= 1`; `pi_hat = 4 * inside.mean()`.
- **8.1** `dist2 = ((X[:,None,:]-C[None,:,:])**2).sum(axis=2)` y luego `labels = dist2.argmin(axis=1)`.
</details>
